# NPY 数值分布分析

这个 Notebook 用于分析一个 `.npy` 文件的数值分布，包括：
- 基础统计量（最小值、最大值、均值、中位数、标准差等）
- 分位数与异常值占比
- 直方图与累计分布（CDF）
- 若是 3D 体数据，展示三个正交切片

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

In [ ]:
# 只需要修改这里：填入你的 npy 文件路径
npy_path = r"../data/合成/x/sample_20260407_165118_354095_t000001_s367119478.npy"

# 直方图参数
num_bins = 120
clip_percentile = (0.5, 99.5)  # 用于“去极值可视化”的分位范围

In [ ]:
if not os.path.exists(npy_path):
    raise FileNotFoundError(f"文件不存在: {npy_path}")

data = np.load(npy_path)
flat = data.reshape(-1)
finite_mask = np.isfinite(flat)
valid = flat[finite_mask]

if valid.size == 0:
    raise ValueError("数据中没有有限值（全是 NaN/Inf）。")

print("=== 数据基本信息 ===")
print(f"路径: {npy_path}")
print(f"shape: {data.shape}")
print(f"dtype: {data.dtype}")
print(f"总元素数: {flat.size:,}")
print(f"有限值数: {valid.size:,} ({valid.size / flat.size:.2%})")
print(f"NaN 数: {np.isnan(flat).sum():,}")
print(f"+Inf 数: {np.isposinf(flat).sum():,}")
print(f"-Inf 数: {np.isneginf(flat).sum():,}")

In [ ]:
def skewness(x):
    m = x.mean()
    s = x.std()
    if s == 0:
        return 0.0
    return np.mean(((x - m) / s) ** 3)

def kurtosis_excess(x):
    m = x.mean()
    s = x.std()
    if s == 0:
        return 0.0
    return np.mean(((x - m) / s) ** 4) - 3.0

pcts = [0, 0.1, 1, 5, 25, 50, 75, 95, 99, 99.9, 100]
qvals = np.percentile(valid, pcts)

print("=== 核心统计量（基于有限值）===")
print(f"min: {valid.min():.6g}")
print(f"max: {valid.max():.6g}")
print(f"mean: {valid.mean():.6g}")
print(f"median: {np.median(valid):.6g}")
print(f"std: {valid.std():.6g}")
print(f"var: {valid.var():.6g}")
print(f"skewness: {skewness(valid):.6g}")
print(f"kurtosis(excess): {kurtosis_excess(valid):.6g}")

print("\n=== 分位数 ===")
for p, v in zip(pcts, qvals):
    print(f"P{p:>5}: {v:.6g}")

# IQR 异常值统计
q1, q3 = np.percentile(valid, [25, 75])
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
outlier_ratio = np.mean((valid < low) | (valid > high))
print("\n=== IQR 异常值统计 ===")
print(f"Q1={q1:.6g}, Q3={q3:.6g}, IQR={iqr:.6g}")
print(f"异常值阈值: [{low:.6g}, {high:.6g}]")
print(f"异常值占比: {outlier_ratio:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 全量直方图
axes[0].hist(valid, bins=num_bins, color="#2E86AB", alpha=0.85)
axes[0].set_title("Histogram (All finite values)")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")

# 去极值直方图（按分位截断，可更清晰看主分布）
lq, uq = np.percentile(valid, clip_percentile)
clipped = valid[(valid >= lq) & (valid <= uq)]
axes[1].hist(clipped, bins=num_bins, color="#F18F01", alpha=0.85)
axes[1].set_title(f"Histogram (Clipped P{clip_percentile[0]}~P{clip_percentile[1]})")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# 累计分布曲线（CDF）
sorted_vals = np.sort(valid)
cdf = np.linspace(0, 1, len(sorted_vals), endpoint=True)

plt.figure(figsize=(8, 5))
plt.plot(sorted_vals, cdf, color="#7B2CBF", linewidth=1.6)
plt.title("Empirical CDF")
plt.xlabel("Value")
plt.ylabel("Cumulative Probability")
plt.tight_layout()
plt.show()

In [ ]:
# 如果是 3D 数据，显示三个正交方向中间切片
if data.ndim == 3:
    i, j, k = data.shape[0] // 2, data.shape[1] // 2, data.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    im0 = axes[0].imshow(data[i, :, :], cmap="gray")
    axes[0].set_title(f"Slice dim0 @ {i}")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(data[:, j, :], cmap="gray")
    axes[1].set_title(f"Slice dim1 @ {j}")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(data[:, :, k], cmap="gray")
    axes[2].set_title(f"Slice dim2 @ {k}")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print(f"当前数据维度是 {data.ndim}，跳过 3D 切片显示。")